In [ ]:
%pip install pandas numpy scipy scikit-learn joblib openpyxl --q

# Backward Elimination — somente Regressão Logística com L1 (por classe)

**Versão reduzida do notebook 4 por classe**: roda **apenas o paradigma LogReg_Lasso**
(Regressão Logística com penalização L1), com exatamente as mesmas funções, a mesma
validação cruzada e o mesmo formato de saída do notebook completo. Serve para gerar ou
refazer só os modelos de regressão logística, sem rodar de novo os outros seis paradigmas.

O que é gravado em cada pasta (`Classe_<rótulo>/` ou a pasta atual):
- `ModelosLogReg_Lasso_CV_Consolidado.xlsx` e `Resumo_Geral_ModelosLogReg_Lasso_CV.xlsx`
  (substituídos, se já existirem);
- `Mapa_Abas_Consolidado.xlsx` e `Alerta_AUC_Abaixo_0.5.xlsx` são **atualizados**: as
  linhas dos outros paradigmas, geradas pelo notebook completo, são mantidas, e só as
  linhas do LogReg_Lasso são substituídas.

---

Para cada arquivo `Ranking_*.xlsx`, este notebook parte da lista completa de atributos
(na ordem do ranking, do melhor para o pior) e remove progressivamente o atributo
pior colocado, avaliando a cada passo um modelo com validação cruzada estratificada
(5-fold). Grava **1 arquivo por paradigma** (`Modelos<Paradigma>_CV_Consolidado.xlsx`),
com **1 aba por ranking**.

**Organização por classe**: o notebook 2 gera os rankings de cada classe numa subpasta própria (`Classe_<rótulo>/`). Este notebook detecta essas subpastas e repete a análise uma vez para cada uma, usando os `Ranking_*.xlsx` de dentro dela e o mesmo alvo multiclasse (a coluna `Classe`, com todas as categorias, lida uma única vez de `base_restante.xlsx` na pasta raiz). O que muda de uma classe para outra é apenas a ordem de eliminação dos atributos, vinda do ranking daquela classe; o classificador sempre prevê a `Classe` completa, não um alvo binário. Se não encontrar nenhuma subpasta `Classe_*/`, processa os `Ranking_*.xlsx` da pasta atual.

Se não encontrar nenhuma subpasta `Classe_*/`, cai de volta no comportamento de
sempre: processa os `Ranking_*.xlsx` da própria pasta atual.

**Atenção ao tempo**: o número de execuções é
`nº de classes × nº de rankings por classe × nº de paradigmas × nº de atributos`
(cada um com 5 folds). Com várias classes, isso multiplica bastante o tempo total em
relação à versão anterior (1 classe só) — ajuste `n_jobs` conforme os núcleos
disponíveis, ou rode uma classe de cada vez comentando as outras ao final do notebook.

### 1. Carregar dados (alvo multiclasse, igual para todas as classes) e descobrir as subpastas

In [ ]:
import pandas as pd
import numpy as np
import os
import time
import json
import re
import warnings
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    confusion_matrix, balanced_accuracy_score, roc_auc_score,
    precision_score, recall_score, f1_score, accuracy_score
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from scipy.stats import wilcoxon
from joblib import Parallel, delayed

warnings.filterwarnings("ignore")

# Base de MODELAGEM (mesma regra do estudo de caso 1): a base restante gerada pelo
# notebook 1 -- linhas que NÃO foram usadas para criar os rankings (notebook 2).
# Escolher os atributos e avaliar os modelos com as mesmas linhas inflaria os
# resultados (viés de seleção); por isso os modelos NÃO usam a amostra.xlsx.
# Lida sempre da RAIZ: o alvo multiclasse (Classe completa) é o mesmo para todas as
# turmas/subpastas; só a ordem de eliminação dos atributos muda entre os cenários.
arquivo_dados = "base_restante.xlsx"
pasta_base = os.getcwd()

df = pd.read_excel(os.path.join(pasta_base, arquivo_dados))
df.columns = df.columns.astype(str).str.strip()

target_column = df.columns[-1]
y = df[target_column].astype(str)
le = LabelEncoder()
y_encoded = le.fit_transform(y)

X_original = df.drop(columns=target_column).select_dtypes(include=[np.number]).copy()
X_original = X_original.fillna(X_original.median())

print(f"Base de modelagem: {arquivo_dados} ({len(df)} linhas)")
print(f"Linhas por classe: {df[target_column].value_counts().to_dict()}")
print(f"Atributos numéricos disponíveis: {X_original.shape[1]}")
print(f"Classes (alvo multiclasse): {list(le.classes_)}")


def descobrir_escopos(pasta_raiz):
    """Descobre as subpastas Classe_<rótulo>/ (saída do notebook 2) dentro de
    `pasta_raiz`. Se nenhuma for encontrada, devolve [(pasta_raiz, None)] -- um único
    escopo, a própria pasta atual (compatibilidade com o fluxo de uma única
    base/classe, como no uso binário original)."""
    subpastas = sorted(
        d for d in os.listdir(pasta_raiz)
        if d.startswith("Classe_") and os.path.isdir(os.path.join(pasta_raiz, d))
    )
    if not subpastas:
        return [(pasta_raiz, None)]
    return [(os.path.join(pasta_raiz, d), d[len("Classe_"):]) for d in subpastas]


ESCOPOS = descobrir_escopos(pasta_base)
if ESCOPOS[0][1] is None:
    print(f"Nenhuma subpasta \'Classe_*\' encontrada -- processando a pasta atual como um único cenário.")
else:
    print(f"{len(ESCOPOS)} subpasta(s) de classe encontrada(s): {[c for _, c in ESCOPOS]}")


### 2. Função de métricas (mesma para todos os paradigmas, e para todas as classes)

In [ ]:
def calcular_metricas(y_true, y_pred, y_proba, le_classes_):
    cm = confusion_matrix(y_true, y_pred, labels=le.transform(le_classes_))
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        sens = tp / (tp + fn) if (tp + fn) > 0 else 0
        espec = tn / (tn + fp) if (tn + fp) > 0 else 0
    else:
        tp, fp, tn, fn, sens, espec = 0, 0, 0, 0, 0, 0

    # AUC: binária usa a probabilidade da classe positiva; multiclasse (nosso caso,
    # 5 classes) usa one-vs-rest com média macro.
    n_classes = y_proba.shape[1]
    if n_classes == 2:
        auc = roc_auc_score(y_true, y_proba[:, 1])
    elif n_classes > 2:
        try:
            auc = roc_auc_score(y_true, y_proba, multi_class="ovr", average="macro")
        except ValueError:
            auc = np.nan
    else:
        auc = np.nan

    return {
        "VP": tp, "FP": fp, "VN": tn, "FN": fn,
        "Sensibilidade": sens, "Especificidade": espec,
        "Acurácia": accuracy_score(y_true, y_pred),
        "Acurácia Balanceada": balanced_accuracy_score(y_true, y_pred),
        "AUC": auc,
        "Precisão (Macro Avg)": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Recall (Macro Avg)": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "F1-Score (Macro Avg)": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "Precisão (Weighted Avg)": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "Recall (Weighted Avg)": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "F1-Score (Weighted Avg)": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }


### 3. Funções genéricas de avaliação (Backward Elimination + CV) — inalteradas

In [ ]:
def avaliar_subset_de_atributos(atributos_usados, X, y, le_classes_, model_factory,
                                 precisa_escalonar=False, n_splits=5):
    import warnings as _w
    _w.filterwarnings("ignore")
    X_subset = X[atributos_usados]
    start_time = time.time()

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    metricas_por_fold = []

    for train_index, test_index in skf.split(X_subset, y):
        X_train, X_test = X_subset.iloc[train_index], X_subset.iloc[test_index]
        y_train, y_test = y[train_index], y[test_index]

        if precisa_escalonar:
            scaler = StandardScaler()
            X_train = scaler.fit_transform(X_train)
            X_test = scaler.transform(X_test)

        modelo = model_factory()
        modelo.fit(X_train, y_train)
        y_pred = modelo.predict(X_test)
        y_proba = modelo.predict_proba(X_test)

        metricas = calcular_metricas(y_test, y_pred, y_proba, le_classes_)
        if metricas:
            metricas_por_fold.append(metricas)

    tempo_gasto = time.time() - start_time

    if not metricas_por_fold:
        return None

    df_metricas = pd.DataFrame(metricas_por_fold)

    resultado_agregado = {
        "Nº de Atributos": len(atributos_usados),
        "Atributos Usados": ", ".join(atributos_usados),
        "Tempo de Execução (s)": round(tempo_gasto, 2),
    }

    for metrica in df_metricas.columns:
        resultado_agregado[f"{metrica} Média"] = round(df_metricas[metrica].mean(), 3)
        resultado_agregado[f"{metrica} Desvio"] = round(df_metricas[metrica].std(), 3)
        resultado_agregado[f"{metrica}_Por_Fold"] = json.dumps(
            [round(float(v), 6) for v in df_metricas[metrica].tolist()]
        )

    resultado_agregado["AUC Abaixo de 0.5"] = bool(resultado_agregado.get("AUC Média", np.nan) < 0.5)

    return resultado_agregado


def valores_por_fold(linha, metrica):
    chave = f"{metrica}_Por_Fold"
    if chave not in linha or pd.isna(linha[chave]):
        return None
    return json.loads(linha[chave])


def teste_wilcoxon_pareado(valores_a, valores_b, alpha=0.05):
    a = np.asarray(valores_a, dtype=float)
    b = np.asarray(valores_b, dtype=float)
    if len(a) != len(b) or len(a) < 1:
        return np.nan, False
    if np.allclose(a, b):
        return 1.0, False
    try:
        _, p_valor = wilcoxon(a, b)
    except ValueError:
        return np.nan, False
    return float(p_valor), bool(p_valor <= alpha)


_CARACTERES_PROIBIDOS_ABA = re.compile(r"[\[\]:*?/\\]")


def nome_aba_excel(nome_desejado, nomes_ja_usados):
    limpo = _CARACTERES_PROIBIDOS_ABA.sub("_", nome_desejado).strip()
    base = limpo[:31] if limpo else "Ranking"
    candidato = base
    contador = 2
    while candidato in nomes_ja_usados:
        sufixo = f"~{contador}"
        candidato = base[: 31 - len(sufixo)] + sufixo
        contador += 1
    nomes_ja_usados.add(candidato)
    return candidato


def rodar_modelo_cv_paralelo(caminho_ranking, model_factory, prefixo, precisa_escalonar=False,
                              n_splits=5, n_jobs=15):
    print(f"\n[{prefixo}] Processando em paralelo ({n_jobs} núcleos): {os.path.basename(caminho_ranking)}")
    try:
        ranking = pd.read_excel(caminho_ranking)
        ranking.columns = ranking.columns.astype(str).str.strip()
        if "Feature" not in ranking.columns:
            print("Sem coluna \'Feature\'. Ignorado.")
            return None, None

        # Quando TODOS os valores de "Feature" são dígitos puros (ex.: nomes de
        # sensor "0","1",...), o pandas relê a coluna como número, não como texto --
        # normaliza para a mesma representação textual usada em X_original.columns.
        # Com os nomes de atributo desta base (textos), este caso não deve ocorrer,
        # mas a defesa é mantida por generalidade (compatibilidade com outras bases).
        def _normalizar_feature(valor):
            if isinstance(valor, float) and valor.is_integer():
                return str(int(valor))
            return str(valor).strip()

        atributos_ordem_remocao = [
            f for f in (_normalizar_feature(a) for a in ranking["Feature"].dropna())
            if f in X_original.columns
        ]
        if not atributos_ordem_remocao:
            print("Ranking vazio ou sem correspondência com as colunas da base. Ignorado.")
            return None, None

        tarefas = [atributos_ordem_remocao[:i] for i in range(len(atributos_ordem_remocao), 0, -1)]

        resultados = Parallel(n_jobs=n_jobs, backend="loky")(
            delayed(avaliar_subset_de_atributos)(
                atributos_usados=subset,
                X=X_original,
                y=y_encoded,
                le_classes_=le.classes_,
                model_factory=model_factory,
                precisa_escalonar=precisa_escalonar,
                n_splits=n_splits,
            ) for subset in tarefas
        )

        resultados = [r for r in resultados if r is not None]
        if not resultados:
            print("Nenhum resultado gerado após o processamento paralelo.")
            return None, None

        nome_ranking = os.path.splitext(os.path.basename(caminho_ranking))[0]
        df_resultados = pd.DataFrame(resultados)

        colunas_iniciais = ["Nº de Atributos", "Atributos Usados", "Tempo de Execução (s)", "AUC Abaixo de 0.5"]
        outras_colunas = [col for col in df_resultados.columns if col not in colunas_iniciais]
        df_resultados = df_resultados[colunas_iniciais + sorted(outras_colunas)]

        n_auc_baixo = int(df_resultados["AUC Abaixo de 0.5"].sum())
        if n_auc_baixo > 0:
            print(f"   {n_auc_baixo} de {len(df_resultados)} linha(s) com AUC Média < 0,50.")

        return nome_ranking, df_resultados

    except Exception as e:
        print(f"Erro ao processar {os.path.basename(caminho_ranking)} com {prefixo}: {e}")
        return None, None


### 4. Paradigma avaliado: Regressão Logística com L1 (os outros seis ficam no notebook completo)

Todos os 7 paradigmas usam os hiperparâmetros **padrão** do scikit-learn, com apenas
`random_state=42` adicionado para reprodutibilidade. Duas exceções, por necessidade
técnica/conceitual, não por preferência de ajuste fino:

- **SVM**: mantém `probability=True` (não é o padrão do sklearn, que é `False`).
  Isso **não é um hiperparâmetro de tuning** -- é exigido porque a Seção 3 chama
  `modelo.predict_proba(X_test)` para todo paradigma (para calcular o AUC); sem
  `probability=True`, o `SVC` não tem `predict_proba` e o notebook quebra nessa
  etapa para o SVM.
- **LogReg_Lasso**: mantém a penalização L1, `solver="liblinear"` e `C=C_LASSO`. Isso
  **não é um hiperparâmetro "a mais"** -- é a própria definição desse paradigma: sem
  a penalização L1, ele deixa de ser uma regressão logística "Lasso" e vira uma
  regressão logística comum.
  **Correção para a base multiclasse**: o `liblinear` só resolve problemas binários, e a
  partir do scikit-learn 1.8 ele dá erro com 3 ou mais classes (*"The 'liblinear' solver
  does not support multiclass classification"*), por isso nenhum modelo era criado. O
  modelo agora é envolvido em `OneVsRestClassifier`, que ajusta um modelo L1 binário por
  classe -- exatamente o que o `liblinear` fazia automaticamente nas versões anteriores.
  O L1 é pedido com `l1_ratio=1.0` no scikit-learn 1.8 ou mais novo e com `penalty="l1"`
  nas versões anteriores (detectado automaticamente).

Para o MLP, `hidden_layer_sizes=(100,)` é o padrão do scikit-learn e `max_iter` assume o padrão de 200. Em alguns subconjuntos de atributos (principalmente os maiores, no início da eliminação) o MLP pode não convergir nesse limite e emitir `ConvergenceWarning`. Isso não interrompe a execução; se necessário, o aviso pode ser evitado passando `max_iter=1000` (ou outro valor) na linha do MLP abaixo.

In [ ]:
from sklearn.multiclass import OneVsRestClassifier
import sklearn

C_LASSO = 1.0  # só usado pelo LogReg_Lasso -- penalização L1 não tem "padrão" no sklearn (ver nota acima)

# Versão do scikit-learn: a partir da 1.8, o L1 é pedido com l1_ratio=1.0 (o parâmetro
# `penalty` foi descontinuado e deixa de existir na 1.10); nas versões anteriores, com
# penalty="l1". Sem esse cuidado, l1_ratio é ignorado nas versões antigas e o modelo vira L2.
_VERSAO_SKL = tuple(int(x) for x in sklearn.__version__.split(".")[:2])
_ARGS_L1 = {"l1_ratio": 1.0} if _VERSAO_SKL >= (1, 8) else {"penalty": "l1"}


def logreg_lasso():
    """Regressão Logística com L1 (Lasso), solver liblinear.

    O liblinear só resolve problemas binários. Até o scikit-learn 1.7, ele tratava a
    base multiclasse sozinho, ajustando um modelo binário por classe (um contra todos,
    one-vs-rest). A partir da 1.8, esse comportamento automático foi retirado e o
    liblinear passou a dar erro com 3 ou mais classes -- foi esse o erro que impediu
    a criação dos modelos. O OneVsRestClassifier faz explicitamente o mesmo que o
    liblinear fazia antes: um modelo L1 binário por classe, com as probabilidades
    normalizadas entre as classes. Com 2 classes, ele ajusta um único modelo, igual
    ao da versão binária do notebook.
    """
    base = LogisticRegression(solver="liblinear", C=C_LASSO, random_state=42, max_iter=1000, **_ARGS_L1)
    return OneVsRestClassifier(base)


PARADIGMAS = {
    "LogReg_Lasso": {"model_factory": logreg_lasso,   # L1 + liblinear, um modelo por classe (ver logreg_lasso)
                      "precisa_escalonar": True},
}

print("Paradigmas configurados:", list(PARADIGMAS.keys()))
print(f"scikit-learn {sklearn.__version__}: L1 definido por {_ARGS_L1}")

# Teste rápido: o LogReg_Lasso precisa ajustar e dar probabilidades para todas as classes
_m = logreg_lasso().fit(StandardScaler().fit_transform(X_original.iloc[:, :10]), y_encoded)
print(f"LogReg_Lasso ok: predict_proba com {_m.predict_proba(StandardScaler().fit_transform(X_original.iloc[:, :10])).shape[1]} "
      f"coluna(s) para {len(le.classes_)} classe(s)")

### 5. Processamento principal — uma vez por subpasta de classe (ou pasta única)

Para cada escopo (`Classe_<rótulo>/` ou a pasta atual, se nenhuma subpasta existir):
roda a Regressão Logística com L1 contra todos os `Ranking_*.xlsx` **daquele escopo**, prevendo
sempre a **Classe completa** (multiclasse) com `X_original`/`y_encoded` — os mesmos
para todas as classes, carregados uma única vez na seção 1. Grava os arquivos
consolidados **dentro da própria subpasta**, exatamente como antes (um
`Modelos<Paradigma>_CV_Consolidado.xlsx` por paradigma, um `Resumo_Geral_*`, um
`Mapa_Abas_Consolidado.xlsx` e um `Alerta_AUC_Abaixo_0.5.xlsx`, se houver).

In [ ]:
for pasta_escopo, classe_rotulo in ESCOPOS:
    rotulo_exibicao = f"Classe {classe_rotulo}" if classe_rotulo else "(pasta única)"
    print(f"\n{'=' * 70}\n {rotulo_exibicao}  ->  {pasta_escopo}\n{'=' * 70}")

    try:
        arquivos_ranking = sorted([
            os.path.join(pasta_escopo, f) for f in os.listdir(pasta_escopo)
            if f.startswith("Ranking_") and f.endswith(".xlsx")
        ])
        if not arquivos_ranking:
            print(f"Nenhum arquivo \'Ranking_*.xlsx\' encontrado em {pasta_escopo}")
    except Exception as e:
        print(f"Erro ao listar arquivos de ranking: {e}")
        arquivos_ranking = []

    print(f"{len(arquivos_ranking)} ranking(s) encontrado(s) para {len(PARADIGMAS)} paradigma(s).")

    resumos_por_paradigma = {prefixo: [] for prefixo in PARADIGMAS}
    auc_baixo_geral = []
    mapa_abas_geral = []

    for prefixo, config in PARADIGMAS.items():
        abas_deste_paradigma = {}
        nomes_abas_usados = set()

        for ranking_path in arquivos_ranking:
            start_geral = time.time()
            nome_ranking, df_resultados = rodar_modelo_cv_paralelo(
                ranking_path,
                model_factory=config["model_factory"],
                prefixo=prefixo,
                precisa_escalonar=config["precisa_escalonar"],
                n_splits=5,
                n_jobs=12,
            )
            end_geral = time.time()
            print(f"--- [{prefixo}] Tempo total para \'{os.path.basename(ranking_path)}\': {end_geral - start_geral:.2f}s ---")

            if df_resultados is None:
                continue

            nome_arquivo_antigo = f"Modelos{prefixo}_CV_{nome_ranking}"
            nome_aba = nome_aba_excel(nome_ranking, nomes_abas_usados)
            abas_deste_paradigma[nome_aba] = df_resultados

            mapa_abas_geral.append({
                "Paradigma": prefixo,
                "Ranking": nome_ranking,
                "Nome Arquivo Antigo": nome_arquivo_antigo,
                "Nome da Aba": nome_aba,
            })

            coluna_principal = "Acurácia Balanceada Média"
            if coluna_principal in df_resultados.columns and not df_resultados.empty:
                idx_melhor = df_resultados[coluna_principal].idxmax()
                melhor = df_resultados.loc[idx_melhor]
                resumos_por_paradigma[prefixo].append({"Ranking": nome_ranking, **melhor.to_dict()})

            if "AUC Abaixo de 0.5" in df_resultados.columns:
                linhas_baixo = df_resultados[df_resultados["AUC Abaixo de 0.5"]]
                for _, linha in linhas_baixo.iterrows():
                    auc_baixo_geral.append({
                        "Paradigma": prefixo, "Ranking": nome_ranking,
                        "Nº de Atributos": linha["Nº de Atributos"],
                        "AUC Média": linha.get("AUC Média", np.nan),
                    })

        if abas_deste_paradigma:
            caminho_consolidado = os.path.join(pasta_escopo, f"Modelos{prefixo}_CV_Consolidado.xlsx")
            with pd.ExcelWriter(caminho_consolidado, engine="openpyxl") as writer:
                for nome_aba, df_aba in abas_deste_paradigma.items():
                    df_aba.to_excel(writer, sheet_name=nome_aba, index=False)
            print(f"\n[{prefixo}] Consolidado salvo: {caminho_consolidado} "
                  f"({len(abas_deste_paradigma)} aba(s)/ranking(s))")
        else:
            print(f"\n[{prefixo}] Nenhum resultado gerado -- consolidado não foi criado.")

    for prefixo, resumo_geral in resumos_por_paradigma.items():
        if resumo_geral:
            df_resumo = pd.DataFrame(resumo_geral)
            resumo_path = os.path.join(pasta_escopo, f"Resumo_Geral_Modelos{prefixo}_CV.xlsx")
            df_resumo.to_excel(resumo_path, index=False)
            print(f"[{prefixo}] Resumo final salvo como: {resumo_path}")
        else:
            print(f"[{prefixo}] Nenhum resultado foi gerado para o resumo final.")

    if mapa_abas_geral:
        df_mapa_abas = pd.DataFrame(mapa_abas_geral)
        path_mapa_abas = os.path.join(pasta_escopo, "Mapa_Abas_Consolidado.xlsx")
        # Mantém as linhas dos outros paradigmas (notebook completo) e troca só as deste
        if os.path.exists(path_mapa_abas):
            antigo = pd.read_excel(path_mapa_abas)
            antigo = antigo[~antigo["Paradigma"].isin(list(PARADIGMAS))]
            df_mapa_abas = pd.concat([antigo, df_mapa_abas], ignore_index=True)
        df_mapa_abas.to_excel(path_mapa_abas, index=False)
        print(f"\n Mapa aba <-> ranking salvo em: {path_mapa_abas}")

    path_auc_baixo = os.path.join(pasta_escopo, "Alerta_AUC_Abaixo_0.5.xlsx")
    antigo_auc = pd.DataFrame()
    if os.path.exists(path_auc_baixo):   # mantém os alertas dos outros paradigmas
        antigo_auc = pd.read_excel(path_auc_baixo)
        antigo_auc = antigo_auc[~antigo_auc["Paradigma"].isin(list(PARADIGMAS))]
    df_auc_baixo = pd.concat([antigo_auc, pd.DataFrame(auc_baixo_geral)], ignore_index=True)
    if not df_auc_baixo.empty:
        df_auc_baixo.sort_values("AUC Média").to_excel(path_auc_baixo, index=False)
    elif os.path.exists(path_auc_baixo):
        os.remove(path_auc_baixo)          # só havia alertas antigos deste paradigma
    if auc_baixo_geral:
        print(f"\n{len(auc_baixo_geral)} combinação(ões) com AUC Média < 0,50 -- ver {path_auc_baixo}")
    else:
        print(f"\nNenhuma combinação com AUC Média abaixo de 0,50 em {rotulo_exibicao}.")

print(f"\nBackward Elimination (Regressão Logística com L1) concluído para todos os escopos.")


### 6. Redundância de variáveis (uma vez por subpasta)

Pergunta: se uma variável é retirada e a acurácia balanceada não cai, ela pode ser considerada redundante?

Para cada escopo, lê o `ModelosLogReg_Lasso_CV_Consolidado.xlsx` **daquele escopo** e
roda o teste de Wilcoxon pareado entre passos consecutivos do Backward Elimination,
gravando `Redundancia_LogReg_Lasso_Por_Combinacao.xlsx` e `Redundancia_LogReg_Lasso_Resumo_Por_Variavel.xlsx`
(nomes próprios, para não sobrescrever os arquivos de redundância do notebook completo)
**dentro da própria subpasta**.

In [ ]:
METRICA_REDUNDANCIA = "Acurácia Balanceada"  # métrica usada para decidir redundância

for pasta_escopo, classe_rotulo in ESCOPOS:
    rotulo_exibicao = f"Classe {classe_rotulo}" if classe_rotulo else "(pasta única)"
    print(f"\n{'=' * 70}\n Redundância -- {rotulo_exibicao}  ->  {pasta_escopo}\n{'=' * 70}")

    arquivos_consolidados = sorted([
        f for f in os.listdir(pasta_escopo)
        if f in {f"Modelos{p}_CV_Consolidado.xlsx" for p in PARADIGMAS}
    ])
    print(f"{len(arquivos_consolidados)} arquivo(s) \'Modelos<Paradigma>_CV_Consolidado.xlsx\' "
          f"encontrado(s) para avaliar redundância.")

    registros_redundancia = []

    for arquivo in arquivos_consolidados:
        m = re.match(r"^Modelos(.+?)_CV_Consolidado\.xlsx$", arquivo)
        if not m:
            print(f"Nome fora do padrão, ignorado: {arquivo}")
            continue
        paradigma = m.group(1)

        todas_as_abas = pd.read_excel(os.path.join(pasta_escopo, arquivo), sheet_name=None)

        for nome_aba, df_modelo in todas_as_abas.items():
            nome_ranking_exibicao = nome_aba
            if f"{METRICA_REDUNDANCIA}_Por_Fold" not in df_modelo.columns:
                print(f"Aba \'{nome_aba}\' de \'{arquivo}\' não tem colunas por fold. Ignorada.")
                continue

            df_modelo = df_modelo.sort_values("Nº de Atributos", ascending=False).reset_index(drop=True)

            for i in range(len(df_modelo) - 1):
                linha_antes = df_modelo.iloc[i]
                linha_depois = df_modelo.iloc[i + 1]

                atributos_antes = set(a.strip() for a in linha_antes["Atributos Usados"].split(","))
                atributos_depois = set(a.strip() for a in linha_depois["Atributos Usados"].split(","))
                removidos = atributos_antes - atributos_depois
                if len(removidos) != 1:
                    print(f"Esperava 1 atributo removido entre N={linha_antes['Nº de Atributos']} e "
                          f"N={linha_depois['Nº de Atributos']} na aba \'{nome_aba}\' de {arquivo}, "
                          f"encontrei {len(removidos)}. Ignorado.")
                    continue
                atributo_removido = removidos.pop()

                folds_antes = valores_por_fold(linha_antes, METRICA_REDUNDANCIA)
                folds_depois = valores_por_fold(linha_depois, METRICA_REDUNDANCIA)
                p_valor, diferenca_sig = teste_wilcoxon_pareado(folds_antes, folds_depois)

                acc_antes = linha_antes[f"{METRICA_REDUNDANCIA} Média"]
                acc_depois = linha_depois[f"{METRICA_REDUNDANCIA} Média"]
                nao_piorou = acc_depois >= acc_antes
                redundante = nao_piorou and not diferenca_sig

                registros_redundancia.append({
                    "Paradigma": paradigma,
                    "Ranking": nome_ranking_exibicao,
                    "Atributo Removido": atributo_removido,
                    "Nº Atributos Antes": linha_antes["Nº de Atributos"],
                    "Nº Atributos Depois": linha_depois["Nº de Atributos"],
                    f"{METRICA_REDUNDANCIA} Antes": acc_antes,
                    f"{METRICA_REDUNDANCIA} Depois": acc_depois,
                    "p-valor (Wilcoxon)": round(p_valor, 4) if not np.isnan(p_valor) else np.nan,
                    "Redundante": redundante,
                })

    if registros_redundancia:
        df_redundancia = pd.DataFrame(registros_redundancia)
        path_red = os.path.join(pasta_escopo, "Redundancia_LogReg_Lasso_Por_Combinacao.xlsx")
        df_redundancia.to_excel(path_red, index=False)
        print(f"{df_redundancia['Redundante'].sum()} de {len(df_redundancia)} remoções marcadas como redundantes.")
        print(f"Salvo: {path_red}")

        df_resumo_var = (
            df_redundancia.groupby("Atributo Removido")
            .agg(Vezes_Avaliada=("Redundante", "size"), Vezes_Redundante=("Redundante", "sum"))
            .reset_index()
        )
        df_resumo_var["Fração Redundante"] = (df_resumo_var["Vezes_Redundante"] / df_resumo_var["Vezes_Avaliada"]).round(3)
        df_resumo_var = df_resumo_var.sort_values("Fração Redundante", ascending=False).reset_index(drop=True)

        path_resumo_var = os.path.join(pasta_escopo, "Redundancia_LogReg_Lasso_Resumo_Por_Variavel.xlsx")
        df_resumo_var.to_excel(path_resumo_var, index=False)
        print(f"Salvo: {path_resumo_var}")
    else:
        print(f"Nenhum arquivo de modelo válido encontrado para avaliar redundância em {rotulo_exibicao}.")

print(f"\nAnálise de redundância concluída para todos os escopos.")
